In [1]:
import os
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.ndimage import zoom
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
TRAJ_DIR   = "Phase3_Data/trajectories/"
IMG_DIR    = "Phase3_Data/generated/experimental/"   # 7000 experimental images
PHASE4_JSON = "Phase4_Results/phase4_results.json"

# Output folders for Phase 7
os.makedirs("Phase7_Results/heatmaps",          exist_ok=True)  # raw 64×64 .npy heatmaps
os.makedirs("Phase7_Results/overlays",          exist_ok=True)  # 512×512 overlay PNGs
os.makedirs("Phase7_Results/gallery",           exist_ok=True)  # qualitative figures

print("All paths ready.")

All paths ready.


In [2]:
with open(PHASE4_JSON) as f:
    records = json.load(f)

print(f"Total records: {len(records)}")
print(f"Example key:   {records[0]['key']}")

# Verify: does key + .npy exist in trajectories folder?
sample_key  = records[0]['key']
sample_file = os.path.join(TRAJ_DIR, sample_key + ".npy")
exists      = os.path.exists(sample_file)
print(f"Trajectory file exists for sample key: {exists}")

# Quick count — how many of the 7000 keys have a matching trajectory file?
missing = [r['key'] for r in records if not os.path.exists(
               os.path.join(TRAJ_DIR, r['key'] + ".npy"))]
print(f"Missing trajectory files: {len(missing)}")  # should be 0

Total records: 7000
Example key:   experimental__SC1__s000__cfg0.0
Trajectory file exists for sample key: True
Missing trajectory files: 0


In [3]:
def compute_spatial_variance(traj_path: str) -> np.ndarray:
    """
    Given a trajectory file of shape (15, 4, 64, 64),
    returns a (64, 64) spatial variance map.

    Steps:
      1. var across axis=0 (15 denoising steps) → (4, 64, 64)
         This tells us: for each channel and each spatial pixel,
         how much did the x0 prediction fluctuate over the window?
      2. mean across axis=0 (4 latent channels) → (64, 64)
         Average over channels gives one scalar per spatial location.
    """
    traj = np.load(traj_path).astype(np.float32)   # (15, 4, 64, 64)
    step_var    = traj.var(axis=0)                  # (4, 64, 64)
    spatial_map = step_var.mean(axis=0)             # (64, 64)
    return spatial_map


def upsample_heatmap(heatmap_64: np.ndarray, target: int = 512) -> np.ndarray:
    """
    Bilinear upsample a (64, 64) map to (target, target).
    zoom factor = 512/64 = 8
    """
    factor = target / heatmap_64.shape[0]
    return zoom(heatmap_64, factor, order=1)   # order=1 = bilinear


def make_overlay(image_pil: Image.Image,
                 heatmap_512: np.ndarray,
                 alpha: float = 0.55) -> Image.Image:
    """
    Blend original image with a colourmap heatmap overlay.
    Returns a PIL Image.
    """
    # Normalise heatmap to [0, 1]
    h_min, h_max = heatmap_512.min(), heatmap_512.max()
    if h_max > h_min:
        heatmap_norm = (heatmap_512 - h_min) / (h_max - h_min)
    else:
        heatmap_norm = np.zeros_like(heatmap_512)

    # Apply 'inferno' colormap (black→red→yellow = low→high variance)
    colormap   = cm.get_cmap('inferno')
    heatmap_rgb = (colormap(heatmap_norm)[:, :, :3] * 255).astype(np.uint8)
    heatmap_img = Image.fromarray(heatmap_rgb).resize(
                      (512, 512), Image.BILINEAR)

    # Blend
    base    = image_pil.convert("RGB").resize((512, 512))
    blended = Image.blend(base, heatmap_img, alpha=alpha)
    return blended


# Quick sanity check on one record
r0   = records[0]
sm   = compute_spatial_variance(os.path.join(TRAJ_DIR, r0['key'] + ".npy"))
sm_up = upsample_heatmap(sm)

print(f"64×64 map  — min: {sm.min():.6f}  max: {sm.max():.6f}  mean: {sm.mean():.6f}")
print(f"512×512 map — shape: {sm_up.shape}")

# Sanity cross-check: mean of spatial map should be close to 
# the global variance already stored in phase4_results
# (They won't be identical because global var was computed differently,
#  but they should be in the same order of magnitude)
print(f"\nSpatial map mean:    {sm.mean():.6f}")
print(f"Stored global var:   {r0['variance']:.6f}")
print(f"Ratio (should be ~1-3x): {sm.mean()/r0['variance']:.2f}")

64×64 map  — min: 0.000109  max: 0.027079  mean: 0.003341
512×512 map — shape: (512, 512)

Spatial map mean:    0.003341
Stored global var:   0.003341
Ratio (should be ~1-3x): 1.00


In [4]:
# Check what the actual image filenames look like
img_files = os.listdir("Phase3_Data/generated/experimental/")
print(f"Total images: {len(img_files)}")
print(f"Example 1: {img_files[0]}")
print(f"Example 2: {img_files[1]}")

# Test the mapping
test_key = records[0]['key']  # "experimental__SC1__s000__cfg0.0"
stripped  = test_key.replace("experimental__", "")
expected_path = f"Phase3_Data/generated/experimental/{stripped}.png"
print(f"\nTest key: {test_key}")
print(f"Expected image path: {expected_path}")
print(f"File exists: {os.path.exists(expected_path)}")

Total images: 7000
Example 1: experimental__AM1__s000__cfg0.0.png
Example 2: experimental__AM1__s000__cfg1.0.png

Test key: experimental__SC1__s000__cfg0.0
Expected image path: Phase3_Data/generated/experimental/SC1__s000__cfg0.0.png
File exists: False


In [5]:
SAVE_OVERLAYS_FOR = {"both", "geometric"}  # only save PNGs for interesting cases

failed = []

for rec in tqdm(records, desc="Computing spatial variance maps"):
    key          = rec['key']
    traj_file    = os.path.join(TRAJ_DIR, key + ".npy")
    failure_type = rec['failure_type']

    # ── 1. Compute spatial variance map ──────────────────────────────────────
    try:
        spatial_map = compute_spatial_variance(traj_file)
    except Exception as e:
        failed.append((key, str(e)))
        continue

    # ── 2. Save raw 64×64 heatmap (all 7,000 images) ─────────────────────────
    np.save(os.path.join("Phase7_Results/heatmaps", key + "_heatmap.npy"),
            spatial_map)

    # ── 3. Save 512×512 overlay PNG (hallucinated + geometric only) ──────────
    if failure_type in SAVE_OVERLAYS_FOR:
        # FIXED: image filename = key + ".png" (no stripping needed)
        img_file = os.path.join(IMG_DIR, key + ".png")

        if os.path.exists(img_file):
            img     = Image.open(img_file)
            hmap_up = upsample_heatmap(spatial_map)
            overlay = make_overlay(img, hmap_up, alpha=0.55)
            overlay.save(os.path.join("Phase7_Results/overlays",
                                      key + "_overlay.png"))
        else:
            failed.append((key, "image file not found"))

print(f"\nDone. Failed: {len(failed)}")
if failed:
    print("Failures:", failed[:5])

Computing spatial variance maps:   0%|          | 0/7000 [00:00<?, ?it/s]C:\Users\Hp\AppData\Local\Temp\ipykernel_50148\2476491925.py:43: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap   = cm.get_cmap('inferno')
Computing spatial variance maps: 100%|██████████| 7000/7000 [03:31<00:00, 33.03it/s]


Done. Failed: 0


In [6]:
heatmap_files = os.listdir("Phase7_Results/heatmaps/")
overlay_files = os.listdir("Phase7_Results/overlays/")

print(f"Heatmap .npy files saved: {len(heatmap_files)}")   # should be 7000
print(f"Overlay .png files saved: {len(overlay_files)}")   # should be 574+364 = 938

# Load one heatmap and check it
sample_hmap = np.load(f"Phase7_Results/heatmaps/{heatmap_files[0]}")
print(f"\nSample heatmap shape: {sample_hmap.shape}")      # (64, 64)
print(f"min: {sample_hmap.min():.6f}  max: {sample_hmap.max():.6f}  mean: {sample_hmap.mean():.6f}")

Heatmap .npy files saved: 7000
Overlay .png files saved: 938

Sample heatmap shape: (64, 64)
min: 0.000109  max: 0.027079  mean: 0.003341


In [7]:
def spatial_concentration_index(heatmap_64: np.ndarray) -> float:
    """
    Measures how concentrated (localized) the variance is.
    
    Method: Gini coefficient of the flattened heatmap values.
    - Gini = 0.0 → perfectly uniform (variance spread everywhere equally)
    - Gini = 1.0 → perfectly concentrated (all variance in one pixel)
    
    Hallucinated images should have HIGHER Gini than clean images
    if mode interpolation is spatially localized.
    """
    flat = heatmap_64.flatten().astype(np.float64)
    flat = np.sort(flat)                          # sort ascending
    n    = len(flat)
    cumsum = np.cumsum(flat)
    # Gini formula
    gini = (2 * np.sum((np.arange(1, n+1) * flat))) / (n * cumsum[-1]) - (n+1)/n
    return float(gini)


def peak_variance_ratio(heatmap_64: np.ndarray, top_pct: float = 0.1) -> float:
    """
    Ratio of mean variance in the top X% pixels 
    vs mean variance in the bottom (1-X)% pixels.
    
    High ratio = variance is concentrated in a small hot region.
    Low ratio  = variance is spread uniformly.
    
    top_pct=0.1 means top 10% of pixels (≈410 pixels out of 4096).
    """
    flat     = heatmap_64.flatten()
    threshold = np.percentile(flat, (1 - top_pct) * 100)
    top_mean  = flat[flat >= threshold].mean()
    bot_mean  = flat[flat <  threshold].mean()
    if bot_mean == 0:
        return 0.0
    return float(top_mean / bot_mean)


# ── Test on one record ────────────────────────────────────────────────────────
sample_hmap = np.load(f"Phase7_Results/heatmaps/{heatmap_files[0]}")
print(f"SCI (Gini):         {spatial_concentration_index(sample_hmap):.4f}")
print(f"Peak/Base ratio:    {peak_variance_ratio(sample_hmap):.4f}")

SCI (Gini):         0.4046
Peak/Base ratio:    3.5678


In [8]:
# Build a lookup: key → heatmap path (fast access)
heatmap_dir   = "Phase7_Results/heatmaps/"
heatmap_lookup = {
    f.replace("_heatmap.npy", ""): os.path.join(heatmap_dir, f)
    for f in os.listdir(heatmap_dir)
}

print(f"Heatmap lookup entries: {len(heatmap_lookup)}")
print(f"Sample lookup key: {list(heatmap_lookup.keys())[0]}")

Heatmap lookup entries: 7000
Sample lookup key: experimental__AM1__s000__cfg0.0


In [9]:
import warnings
warnings.filterwarnings("ignore")

spatial_metrics = []   # will become our analysis dataframe

for rec in tqdm(records, desc="Computing spatial metrics"):
    key  = rec['key']
    hmap = np.load(heatmap_lookup[key])

    sci  = spatial_concentration_index(hmap)
    pvr  = peak_variance_ratio(hmap, top_pct=0.10)

    # Also store: where is the peak variance region (for YOLO overlap later)
    # Peak region = pixels above 90th percentile of this heatmap
    threshold_90 = np.percentile(hmap, 90)
    peak_mask    = (hmap >= threshold_90).astype(np.uint8)  # 64×64 binary mask
    peak_frac    = peak_mask.mean()   # always ~0.10 by construction

    spatial_metrics.append({
        "key"           : key,
        "prompt_id"     : rec['prompt_id'],
        "category"      : rec['category'],
        "cfg"           : rec['cfg'],
        "failure_type"  : rec['failure_type'],
        "hallucinated"  : rec['hallucinated'],
        "global_var"    : rec['variance'],
        "sci_gini"      : sci,
        "peak_var_ratio": pvr,
        "spatial_mean"  : float(hmap.mean()),
        "spatial_max"   : float(hmap.max()),
        "spatial_std"   : float(hmap.std()),
    })

print(f"Done. Total records: {len(spatial_metrics)}")

Computing spatial metrics: 100%|██████████| 7000/7000 [03:25<00:00, 34.08it/s]

Done. Total records: 7000


In [13]:
from scipy import stats
import math

# Split by hallucinated vs clean
hall_sci  = [m['sci_gini']       for m in spatial_metrics if m['hallucinated']]
clean_sci = [m['sci_gini']       for m in spatial_metrics if not m['hallucinated']]
hall_pvr  = [m['peak_var_ratio'] for m in spatial_metrics if m['hallucinated']]
clean_pvr = [m['peak_var_ratio'] for m in spatial_metrics if not m['hallucinated']]

print(f"Hallucinated (n={len(hall_sci)})  — SCI mean: {np.mean(hall_sci):.4f}  std: {np.std(hall_sci):.4f}")
print(f"Clean        (n={len(clean_sci)}) — SCI mean: {np.mean(clean_sci):.4f}  std: {np.std(clean_sci):.4f}")
print(f"\nHallucinated PVR mean: {np.mean(hall_pvr):.4f}")
print(f"Clean        PVR mean: {np.mean(clean_pvr):.4f}")

# CORRECTED: alternative='less' because hallucinated SCI < clean SCI
mw_sci = stats.mannwhitneyu(hall_sci, clean_sci, alternative='less')
mw_pvr = stats.mannwhitneyu(hall_pvr, clean_pvr, alternative='less')

n_total = len(hall_sci) + len(clean_sci)

# Effect size r — use absolute Z from two-sided test for correct magnitude
mw_sci_2 = stats.mannwhitneyu(hall_sci, clean_sci, alternative='two-sided')
mw_pvr_2 = stats.mannwhitneyu(hall_pvr, clean_pvr, alternative='two-sided')

# r = Z / sqrt(N),  Z from two-sided p
z_sci = abs(stats.norm.ppf(mw_sci_2.pvalue / 2))
z_pvr = abs(stats.norm.ppf(mw_pvr_2.pvalue / 2))
r_sci = z_sci / math.sqrt(n_total)
r_pvr = z_pvr / math.sqrt(n_total)

def effect_label(r):
    r = abs(r)
    if r > 0.5:   return "(Large)"
    if r > 0.3:   return "(Medium)"
    return "(Small)"

print(f"\n── Spatial Concentration Index (Gini) ──")
print(f"Direction: hallucinated SCI < clean SCI  ← variance more UNIFORM in hallucinated")
print(f"Mann-Whitney U = {mw_sci.statistic:.0f},  p = {mw_sci.pvalue:.3e}")
print(f"Effect size r  = {r_sci:.3f}  {effect_label(r_sci)}")

print(f"\n── Peak Variance Ratio (top 10%) ──")
print(f"Direction: hallucinated PVR < clean PVR  ← no dominant hot-spot in hallucinated")
print(f"Mann-Whitney U = {mw_pvr.statistic:.0f},  p = {mw_pvr.pvalue:.3e}")
print(f"Effect size r  = {r_pvr:.3f}  {effect_label(r_pvr)}")

# Cohen's d for SCI (for report consistency with Phase 5)
def cohens_d(a, b):
    pooled_std = math.sqrt((np.std(a)**2 + np.std(b)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std

d_sci = cohens_d(hall_sci, clean_sci)
d_pvr = cohens_d(hall_pvr, clean_pvr)
print(f"\n── Cohen's d ──")
print(f"SCI: d = {d_sci:.4f}  {effect_label(abs(d_sci))}")
print(f"PVR: d = {d_pvr:.4f}  {effect_label(abs(d_pvr))}")

Hallucinated (n=574)  — SCI mean: 0.3858  std: 0.0374
Clean        (n=6426) — SCI mean: 0.4358  std: 0.0588

Hallucinated PVR mean: 3.3329
Clean        PVR mean: 4.1653

── Spatial Concentration Index (Gini) ──
Direction: hallucinated SCI < clean SCI  ← variance more UNIFORM in hallucinated
Mann-Whitney U = 648503,  p = 8.084e-147
Effect size r  = 0.308  (Medium)

── Peak Variance Ratio (top 10%) ──
Direction: hallucinated PVR < clean PVR  ← no dominant hot-spot in hallucinated
Mann-Whitney U = 641923,  p = 2.056e-148
Effect size r  = 0.310  (Medium)

── Cohen's d ──
SCI: d = -1.0136  (Large)
PVR: d = -0.7625  (Large)


In [14]:
categories = ["Single-Concept", "Compositional", "Negation", "Ambiguous", "Conflicting"]

print(f"{'Category':<20} {'Hall SCI':>10} {'Clean SCI':>10} {'Diff':>8} {'p-value':>12} {'Interpretation':>20}")
print("─" * 85)

for cat in categories:
    h = [m['sci_gini'] for m in spatial_metrics
         if m['category'] == cat and m['hallucinated']]
    c = [m['sci_gini'] for m in spatial_metrics
         if m['category'] == cat and not m['hallucinated']]

    if len(h) < 2:
        print(f"{cat:<20} {'N/A':>10} {np.mean(c):>10.4f} {'—':>8} {'N/A':>12} {'No hallucinations':>20}")
        continue

    # Corrected direction: hallucinated < clean
    mw   = stats.mannwhitneyu(h, c, alternative='less')
    diff = np.mean(h) - np.mean(c)
    sig  = "***" if mw.pvalue < 0.001 else "**" if mw.pvalue < 0.01 else "*" if mw.pvalue < 0.05 else "ns"
    print(f"{cat:<20} {np.mean(h):>10.4f} {np.mean(c):>10.4f} {diff:>+8.4f} {mw.pvalue:>12.3e} {sig:>20}")

Category               Hall SCI  Clean SCI     Diff      p-value       Interpretation
─────────────────────────────────────────────────────────────────────────────────────
Single-Concept           0.3829     0.4479  -0.0650    6.512e-89                  ***
Compositional            0.3942     0.4637  -0.0695    1.386e-74                  ***
Negation                 0.3729     0.4135  -0.0407    7.748e-28                  ***
Ambiguous                   N/A     0.4222        —          N/A    No hallucinations
Conflicting                 N/A     0.4368        —          N/A    No hallucinations


In [15]:
import json

with open("Phase7_Results/spatial_metrics.json", "w") as f:
    json.dump(spatial_metrics, f, indent=2)

print(f"Saved {len(spatial_metrics)} records to Phase7_Results/spatial_metrics.json")

# Quick summary
hall_records = [m for m in spatial_metrics if m['hallucinated']]
clean_records = [m for m in spatial_metrics if not m['hallucinated']]

print(f"\nSummary:")
print(f"  Hallucinated: {len(hall_records)}")
print(f"  Clean:        {len(clean_records)}")
print(f"\n  Hallucinated — mean SCI: {np.mean([m['sci_gini'] for m in hall_records]):.4f}")
print(f"  Clean        — mean SCI: {np.mean([m['sci_gini'] for m in clean_records]):.4f}")
print(f"\n  Hallucinated — mean PVR: {np.mean([m['peak_var_ratio'] for m in hall_records]):.4f}")
print(f"  Clean        — mean PVR: {np.mean([m['peak_var_ratio'] for m in clean_records]):.4f}")

Saved 7000 records to Phase7_Results/spatial_metrics.json

Summary:
  Hallucinated: 574
  Clean:        6426

  Hallucinated — mean SCI: 0.3858
  Clean        — mean SCI: 0.4358

  Hallucinated — mean PVR: 3.3329
  Clean        — mean PVR: 4.1653


In [16]:
# ── Select representative images for the gallery ─────────────────────────────
# Strategy: pick the clearest examples from each failure type and category
# For hallucinated: highest global variance (most extreme mode interpolation)
# For clean: highest SCI (most concentrated — good contrast visual)

import json

# Reload for easy filtering
with open("Phase7_Results/spatial_metrics.json") as f:
    smetrics = json.load(f)

# Also build a lookup from phase4 records for prompt text
prompt_lookup = {r['key']: r['prompt'] for r in records}

# ── 1. Top hallucinated per category (by global variance) ────────────────────
hall_by_cat = {}
for cat in ["Single-Concept", "Compositional", "Negation"]:
    candidates = [m for m in smetrics 
                  if m['category'] == cat and m['failure_type'] == 'both']
    candidates.sort(key=lambda x: x['global_var'], reverse=True)
    hall_by_cat[cat] = candidates[:3]   # top 3 per category

# ── 2. Geometric-only (high variance but clean output) ───────────────────────
geo_only = [m for m in smetrics if m['failure_type'] == 'geometric']
geo_only.sort(key=lambda x: x['global_var'], reverse=True)
geo_top = geo_only[:3]

# ── 3. Clean images with highest SCI (for contrast) ─────────────────────────
clean_high_sci = [m for m in smetrics if m['failure_type'] == 'none']
clean_high_sci.sort(key=lambda x: x['sci_gini'], reverse=True)
clean_top = clean_high_sci[:3]

# Print selection summary
print("── Hallucinated (top 3 per category) ──")
for cat, items in hall_by_cat.items():
    for m in items:
        print(f"  [{cat}] {m['key']}  var={m['global_var']:.5f}  "
              f"SCI={m['sci_gini']:.4f}  prompt: '{prompt_lookup[m['key']]}'")

print("\n── Geometric-only (top 3) ──")
for m in geo_top:
    print(f"  {m['key']}  var={m['global_var']:.5f}  "
          f"SCI={m['sci_gini']:.4f}  prompt: '{prompt_lookup[m['key']]}'")

print("\n── Clean high-SCI (top 3) ──")
for m in clean_top:
    print(f"  {m['key']}  var={m['global_var']:.5f}  "
          f"SCI={m['sci_gini']:.4f}  prompt: '{prompt_lookup[m['key']]}'")

── Hallucinated (top 3 per category) ──
  [Single-Concept] experimental__SC1__s040__cfg0.0  var=0.00420  SCI=0.3888  prompt: 'A single red apple'
  [Single-Concept] experimental__SC2__s040__cfg0.0  var=0.00420  SCI=0.3888  prompt: 'A blue coffee mug'
  [Single-Concept] experimental__SC3__s040__cfg0.0  var=0.00420  SCI=0.3888  prompt: 'A yellow sunflower'
  [Compositional] experimental__CO1__s008__cfg15.0  var=0.00537  SCI=0.5616  prompt: 'Cat on chair'
  [Compositional] experimental__CO1__s032__cfg15.0  var=0.00511  SCI=0.5242  prompt: 'Cat on chair'
  [Compositional] experimental__CO1__s002__cfg15.0  var=0.00499  SCI=0.6194  prompt: 'Cat on chair'
  [Negation] experimental__NE2__s042__cfg1.0  var=0.00441  SCI=0.3415  prompt: 'Face, no eyes'
  [Negation] experimental__NE1__s040__cfg0.0  var=0.00420  SCI=0.3888  prompt: 'Car, no wheels'
  [Negation] experimental__NE2__s040__cfg0.0  var=0.00420  SCI=0.3888  prompt: 'Face, no eyes'

── Geometric-only (top 3) ──
  experimental__CF3__s040__

In [19]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — avoids display issues
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import colormaps
import numpy as np
from PIL import Image
from scipy.ndimage import zoom

def load_components(key, img_dir, heatmap_dir):
    """Load original image + heatmap for a given key."""
    img_path  = os.path.join(img_dir, key + ".png")
    hmap_path = os.path.join(heatmap_dir, key + "_heatmap.npy")
    
    img  = Image.open(img_path).convert("RGB").resize((256, 256))
    hmap = np.load(hmap_path)                      # (64, 64)
    hmap_up = zoom(hmap, 256/64, order=1)          # (256, 256)
    
    # Normalize heatmap globally within this image
    h_min, h_max = hmap_up.min(), hmap_up.max()
    hmap_norm = (hmap_up - h_min) / (h_max - h_min + 1e-10)
    
    return np.array(img), hmap_norm

HEATMAP_DIR = "Phase7_Results/heatmaps/"
IMG_DIR_EXP = "Phase3_Data/generated/experimental/"
cmap = colormaps['inferno']

# ── Figure layout: 3 rows (Hall / Geo / Clean), each row = 3 examples ────────
# Each example = 2 panels (original | heatmap)
# Total: 3 rows × 3 examples × 2 panels = 18 subplots

fig = plt.figure(figsize=(20, 13))
fig.patch.set_facecolor('#0f0f0f')

row_configs = [
    ("HALLUCINATED (Both: High Variance + Quality Failure)",
     [m for cat_items in hall_by_cat.values() for m in cat_items[:1]]
     + [list(hall_by_cat.values())[1][1]],   # 3 items: 1 per category
     "#ff4444"),
    ("GEOMETRIC ONLY (High Variance, Output Correct)",
     geo_top[:3],
     "#ff8c00"),
    ("CLEAN (Low Variance, Correct Output)",
     clean_top[:3],
     "#44ff88"),
]

# Rebuild row_configs cleanly with exactly 3 items per row
row_data = [
    ("HALLUCINATED — mode interpolation confirmed",
     [list(hall_by_cat.values())[0][0],
      list(hall_by_cat.values())[1][0],
      list(hall_by_cat.values())[2][0]],
     "#ff4444"),
    ("GEOMETRIC ONLY — high variance, output still correct",
     geo_top[:3],
     "#ff9900"),
    ("CLEAN — low uniform variance, successful generation",
     clean_top[:3],
     "#44cc88"),
]

for row_idx, (row_title, items, title_color) in enumerate(row_data):
    for col_idx, m in enumerate(items):
        key = m['key']
        try:
            img_arr, hmap_norm = load_components(key, IMG_DIR_EXP, HEATMAP_DIR)
        except Exception as e:
            print(f"Skip {key}: {e}")
            continue

        # Original image panel
        ax_img = fig.add_subplot(3, 6, row_idx*6 + col_idx*2 + 1)
        ax_img.imshow(img_arr)
        ax_img.set_xticks([]); ax_img.set_yticks([])

        prompt_short = prompt_lookup[key][:28] + "…" if len(prompt_lookup[key]) > 28 \
                       else prompt_lookup[key]
        ax_img.set_title(
            f"CFG={m['cfg']}  seed={m['key'].split('__s')[1].split('__')[0]}\n"
            f"\"{prompt_short}\"",
            fontsize=7, color='white', pad=3
        )
        for spine in ax_img.spines.values():
            spine.set_edgecolor(title_color); spine.set_linewidth(2)

        # Heatmap panel
        ax_hm = fig.add_subplot(3, 6, row_idx*6 + col_idx*2 + 2)
        ax_hm.imshow(hmap_norm, cmap='inferno', vmin=0, vmax=1)
        ax_hm.set_xticks([]); ax_hm.set_yticks([])
        ax_hm.set_title(
            f"SCI={m['sci_gini']:.3f}  PVR={m['peak_var_ratio']:.2f}\n"
            f"var={m['global_var']:.5f}",
            fontsize=7, color='#ffcc44', pad=3
        )
        for spine in ax_hm.spines.values():
            spine.set_edgecolor('#555555'); spine.set_linewidth(1)

    # Row label on left
    fig.text(
        0.01, 1 - (row_idx + 0.5) / 3,
        row_title,
        va='center', ha='left', fontsize=9,
        color=title_color, fontweight='bold',
        rotation=90
    )

# Colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='inferno',
                            norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Normalised Spatial Variance', color='white', fontsize=9)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=8)

fig.suptitle(
    "Phase 7 — Spatial Variance Heatmaps by Failure Type\n"
    "Hallucinated images show uniform variance spread; "
    "Clean images show localized hot spots",
    fontsize=12, color='white', y=0.98, fontweight='bold'
)

plt.tight_layout(rect=[0.04, 0.01, 0.91, 0.96])
plt.savefig("Phase7_Results/gallery/figure5_spatial_gallery.png",
            dpi=150, bbox_inches='tight',
            facecolor='#0f0f0f')
plt.close()
print("Figure 5 saved → Phase7_Results/gallery/figure5_spatial_gallery.png")

Figure 5 saved → Phase7_Results/gallery/figure5_spatial_gallery.png


In [20]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a1a')

hall_sci_vals  = [m['sci_gini'] for m in smetrics if m['hallucinated']]
clean_sci_vals = [m['sci_gini'] for m in smetrics if not m['hallucinated']]
geo_sci_vals   = [m['sci_gini'] for m in smetrics if m['failure_type'] == 'geometric']
sem_sci_vals   = [m['sci_gini'] for m in smetrics if m['failure_type'] == 'semantic']

# ── Left: KDE distributions ───────────────────────────────────────────────────
from scipy.stats import gaussian_kde

colors = {'hallucinated': '#ff4444', 'geometric': '#ff9900',
          'semantic': '#4488ff',     'clean': '#44cc88'}
groups = [
    (hall_sci_vals,  'Hallucinated (both)',   '#ff4444'),
    (geo_sci_vals,   'Geometric only',        '#ff9900'),
    (sem_sci_vals,   'Semantic only',         '#4488ff'),
    (clean_sci_vals, 'Clean',                 '#44cc88'),
]

x_range = np.linspace(0.15, 0.65, 300)
for vals, label, color in groups:
    if len(vals) < 5:
        continue
    kde  = gaussian_kde(vals, bw_method=0.15)
    axes[0].fill_between(x_range, kde(x_range),
                         alpha=0.25, color=color)
    axes[0].plot(x_range, kde(x_range),
                 color=color, linewidth=2, label=f"{label} (n={len(vals)})")

axes[0].axvline(np.mean(hall_sci_vals),  color='#ff4444', ls='--', lw=1.5,
                label=f"Hall. mean={np.mean(hall_sci_vals):.3f}")
axes[0].axvline(np.mean(clean_sci_vals), color='#44cc88', ls='--', lw=1.5,
                label=f"Clean mean={np.mean(clean_sci_vals):.3f}")

axes[0].set_xlabel('Spatial Concentration Index (Gini)', color='white', fontsize=11)
axes[0].set_ylabel('Density', color='white', fontsize=11)
axes[0].set_title('SCI Distribution by Failure Type\n'
                  f'Cohen\'s d = {d_sci:.3f} (Large effect)',
                  color='white', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white')
axes[0].tick_params(colors='white')
for spine in axes[0].spines.values():
    spine.set_edgecolor('#444444')

# ── Right: Per-category mean SCI (hallucinated vs clean) ─────────────────────
cats_with_hall = ["Single-Concept", "Compositional", "Negation"]
x = np.arange(len(cats_with_hall))
width = 0.35

hall_means  = [np.mean([m['sci_gini'] for m in smetrics
                         if m['category']==c and m['hallucinated']])
               for c in cats_with_hall]
clean_means = [np.mean([m['sci_gini'] for m in smetrics
                         if m['category']==c and not m['hallucinated']])
               for c in cats_with_hall]
hall_stds   = [np.std([m['sci_gini'] for m in smetrics
                        if m['category']==c and m['hallucinated']])
               for c in cats_with_hall]
clean_stds  = [np.std([m['sci_gini'] for m in smetrics
                        if m['category']==c and not m['hallucinated']])
               for c in cats_with_hall]

b1 = axes[1].bar(x - width/2, hall_means,  width, yerr=hall_stds,
                  color='#ff4444', alpha=0.8, capsize=4,
                  label='Hallucinated', error_kw={'color': 'white'})
b2 = axes[1].bar(x + width/2, clean_means, width, yerr=clean_stds,
                  color='#44cc88', alpha=0.8, capsize=4,
                  label='Clean', error_kw={'color': 'white'})

axes[1].set_xticks(x)
axes[1].set_xticklabels(cats_with_hall, color='white', fontsize=10)
axes[1].set_ylabel('Mean SCI (Gini)', color='white', fontsize=11)
axes[1].set_title('Per-Category SCI: Hallucinated vs Clean\n'
                  'All differences p < 0.001 (***)',
                  color='white', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9, facecolor='#2a2a2a', labelcolor='white')
axes[1].tick_params(colors='white')
for spine in axes[1].spines.values():
    spine.set_edgecolor('#444444')

# Significance stars
for i in range(len(cats_with_hall)):
    y_max = max(hall_means[i] + hall_stds[i],
                clean_means[i] + clean_stds[i]) + 0.005
    axes[1].text(i, y_max, '***', ha='center',
                 color='white', fontsize=13, fontweight='bold')

fig.suptitle('Phase 7 — Spatial Concentration Index Analysis\n'
             'Hallucinated images have MORE UNIFORM variance (lower SCI)',
             color='white', fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig("Phase7_Results/gallery/figure6_sci_distributions.png",
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.close()
print("Figure 6 saved → Phase7_Results/gallery/figure6_sci_distributions.png")

Figure 6 saved → Phase7_Results/gallery/figure6_sci_distributions.png


In [21]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Build feature arrays ──────────────────────────────────────────────────────
y      = np.array([1 if m['hallucinated'] else 0 for m in smetrics])

# Phase 5 signals (already had these)
X_gvar  = np.array([m['global_var']     for m in smetrics])   # global variance
X_clip  = np.array([m['sci_gini']       for m in smetrics])   # reusing name — 
# Note: clip score not in spatial_metrics, load from phase4 records
clip_lookup = {r['key']: r['clip_score'] for r in records}
X_clipsc = np.array([clip_lookup[m['key']] for m in smetrics])

# Phase 7 spatial signals
X_sci  = np.array([m['sci_gini']        for m in smetrics])
X_pvr  = np.array([m['peak_var_ratio']  for m in smetrics])
X_smax = np.array([m['spatial_max']     for m in smetrics])
X_sstd = np.array([m['spatial_std']     for m in smetrics])

# ── Compute AUC for each signal ───────────────────────────────────────────────
# Note directionality: 
#   global_var  → higher = more hallucinated  (use as-is)
#   clip_score  → lower  = more hallucinated  (negate)
#   sci_gini    → lower  = more hallucinated  (negate)
#   pvr         → lower  = more hallucinated  (negate)
#   spatial_max → higher = more hallucinated  (use as-is)

signals = [
    ("Global Variance (Phase 5)",    X_gvar,       False),
    ("CLIP Score (Phase 5)",         X_clipsc,     True),   # True = negate
    ("SCI / Gini (Phase 7)",         X_sci,        True),
    ("Peak Variance Ratio (Phase 7)",X_pvr,        True),
    ("Spatial Max (Phase 7)",        X_smax,       False),
    ("Spatial Std (Phase 7)",        X_sstd,       False),
]

print(f"{'Signal':<35} {'AUC':>8}  {'Interpretation'}")
print("─" * 70)

auc_results = {}
for name, X, negate in signals:
    X_use = -X if negate else X
    auc = roc_auc_score(y, X_use)
    auc_results[name] = auc
    flag = "★ BEST" if auc == max(roc_auc_score(y, -x if neg else x) 
                                   for _, x, neg in signals) else ""
    interp = "Excellent" if auc>0.90 else "Good" if auc>0.80 else "Fair" if auc>0.70 else "Poor"
    print(f"{name:<35} {auc:>8.4f}  {interp}")

# ── Combined: Phase 5 features only ──────────────────────────────────────────
scaler5 = StandardScaler()
X5 = scaler5.fit_transform(np.column_stack([X_gvar, X_clipsc]))
lr5 = LogisticRegression(random_state=42, max_iter=1000)
cv5 = cross_val_score(lr5, X5, y, cv=5, scoring='roc_auc')
lr5.fit(X5, y)
auc5_combined = roc_auc_score(y, lr5.predict_proba(X5)[:,1])

# ── Combined: Phase 7 spatial features only ───────────────────────────────────
scaler7 = StandardScaler()
X7 = scaler7.fit_transform(np.column_stack([X_gvar, X_sci, X_pvr, X_smax, X_sstd]))
lr7 = LogisticRegression(random_state=42, max_iter=1000)
cv7 = cross_val_score(lr7, X7, y, cv=5, scoring='roc_auc')
lr7.fit(X7, y)
auc7_combined = roc_auc_score(y, lr7.predict_proba(X7)[:,1])

# ── Combined: All features (Phase 5 + Phase 7) ────────────────────────────────
scalerAll = StandardScaler()
XAll = scalerAll.fit_transform(
    np.column_stack([X_gvar, X_clipsc, X_sci, X_pvr, X_smax, X_sstd]))
lrAll = LogisticRegression(random_state=42, max_iter=1000)
cvAll = cross_val_score(lrAll, XAll, y, cv=5, scoring='roc_auc')
lrAll.fit(XAll, y)
aucAll_combined = roc_auc_score(y, lrAll.predict_proba(XAll)[:,1])

print(f"\n{'─'*70}")
print(f"{'Phase 5 combined (Var + CLIP)':<35} {auc5_combined:>8.4f}  "
      f"5-fold CV: {cv5.mean():.4f}±{cv5.std():.4f}")
print(f"{'Phase 7 spatial combined':<35} {auc7_combined:>8.4f}  "
      f"5-fold CV: {cv7.mean():.4f}±{cv7.std():.4f}")
print(f"{'All features combined':<35} {aucAll_combined:>8.4f}  "
      f"5-fold CV: {cvAll.mean():.4f}±{cvAll.std():.4f}")

print(f"\nPhase 5 LR coefficients:")
print(f"  Global Var: {lr5.coef_[0][0]:+.3f}   CLIP: {lr5.coef_[0][1]:+.3f}")
print(f"\nAll-features LR coefficients:")
feat_names = ["GlobalVar","CLIP","SCI","PVR","SpatMax","SpatStd"]
for name, coef in zip(feat_names, lrAll.coef_[0]):
    print(f"  {name:<12}: {coef:+.3f}")

Signal                                   AUC  Interpretation
──────────────────────────────────────────────────────────────────────
Global Variance (Phase 5)             0.8766  Good
CLIP Score (Phase 5)                  0.8725  Good
SCI / Gini (Phase 7)                  0.8242  Good
Peak Variance Ratio (Phase 7)         0.8260  Good
Spatial Max (Phase 7)                 0.4211  Poor
Spatial Std (Phase 7)                 0.6560  Poor

──────────────────────────────────────────────────────────────────────
Phase 5 combined (Var + CLIP)         0.8979  5-fold CV: 0.8962±0.0478
Phase 7 spatial combined              0.8994  5-fold CV: 0.9000±0.0796
All features combined                 0.9258  5-fold CV: 0.9185±0.0622

Phase 5 LR coefficients:
  Global Var: +0.526   CLIP: -1.417

All-features LR coefficients:
  GlobalVar   : +2.844
  CLIP        : -1.098
  SCI         : -0.661
  PVR         : +1.561
  SpatMax     : -0.988
  SpatStd     : -3.792


In [22]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes: ax.set_facecolor('#1a1a1a')

# ── Left panel: individual signal ROC curves ──────────────────────────────────
curve_configs = [
    ("Global Variance",    X_gvar,   False, '#ff4444', '--'),
    ("CLIP Score",         X_clipsc, True,  '#4488ff', '--'),
    ("SCI / Gini",         X_sci,    True,  '#ff9900', '-'),
    ("Peak Var Ratio",     X_pvr,    True,  '#aa44ff', '-'),
    ("Spatial Max",        X_smax,   False, '#44ffcc', '-'),
]

for name, X, negate, color, ls in curve_configs:
    X_use = -X if negate else X
    fpr, tpr, _ = roc_curve(y, X_use)
    auc = roc_auc_score(y, X_use)
    axes[0].plot(fpr, tpr, color=color, lw=2, ls=ls,
                 label=f"{name}  AUC={auc:.4f}")

axes[0].plot([0,1],[0,1], color='#555555', lw=1, ls=':',
             label='Random (AUC=0.500)')
axes[0].set_xlabel('False Positive Rate', color='white', fontsize=11)
axes[0].set_ylabel('True Positive Rate', color='white', fontsize=11)
axes[0].set_title('Individual Signal ROC Curves\n'
                  'Dashed = Phase 5 signals, Solid = Phase 7 spatial',
                  color='white', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white',
               loc='lower right')
axes[0].tick_params(colors='white')
for spine in axes[0].spines.values(): spine.set_edgecolor('#444444')

# ── Right panel: combined model comparison ────────────────────────────────────
combined_configs = [
    ("Phase 5: Var + CLIP",    X5,    lr5,    cv5,    '#4488ff'),
    ("Phase 7: Spatial Only",  X7,    lr7,    cv7,    '#ff9900'),
    ("All Features Combined",  XAll,  lrAll,  cvAll,  '#44ff88'),
]

for name, X_feat, lr_model, cv_scores, color in combined_configs:
    fpr, tpr, _ = roc_curve(y, lr_model.predict_proba(X_feat)[:,1])
    auc = roc_auc_score(y, lr_model.predict_proba(X_feat)[:,1])
    axes[1].fill_between(fpr, tpr, alpha=0.08, color=color)
    axes[1].plot(fpr, tpr, color=color, lw=2.5,
                 label=f"{name}\nAUC={auc:.4f}  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}")

axes[1].plot([0,1],[0,1], color='#555555', lw=1, ls=':')
axes[1].set_xlabel('False Positive Rate', color='white', fontsize=11)
axes[1].set_ylabel('True Positive Rate', color='white', fontsize=11)
axes[1].set_title('Combined Model ROC Curves\n'
                  'Does spatial information improve over Phase 5?',
                  color='white', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8.5, facecolor='#2a2a2a', labelcolor='white',
               loc='lower right')
axes[1].tick_params(colors='white')
for spine in axes[1].spines.values(): spine.set_edgecolor('#444444')

fig.suptitle('Phase 7 — ROC-AUC Comparison: Spatial vs Global Signals',
             color='white', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig("Phase7_Results/gallery/figure7_roc_comparison.png",
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.close()
print("Figure 7 saved → Phase7_Results/gallery/figure7_roc_comparison.png")

Figure 7 saved → Phase7_Results/gallery/figure7_roc_comparison.png


In [23]:
import json

phase7_summary = {
    "spatial_concentration": {
        "hall_sci_mean"  : float(np.mean(hall_sci_vals)),
        "clean_sci_mean" : float(np.mean(clean_sci_vals)),
        "geo_sci_mean"   : float(np.mean([m['sci_gini'] for m in smetrics 
                                          if m['failure_type']=='geometric'])),
        "cohens_d"       : float(d_sci),
        "mannwhitney_p"  : float(mw_sci.pvalue),
        "effect_r"       : float(r_sci),
    },
    "peak_variance_ratio": {
        "hall_pvr_mean"  : float(np.mean(hall_pvr)),
        "clean_pvr_mean" : float(np.mean(clean_pvr)),
        "cohens_d"       : float(d_pvr),
        "mannwhitney_p"  : float(mw_pvr.pvalue),
        "effect_r"       : float(r_pvr),
    },
    "auc_comparison": {
        "global_variance_auc"   : float(auc_results["Global Variance (Phase 5)"]),
        "clip_score_auc"        : float(auc_results["CLIP Score (Phase 5)"]),
        "sci_gini_auc"          : float(auc_results["SCI / Gini (Phase 7)"]),
        "peak_var_ratio_auc"    : float(auc_results["Peak Variance Ratio (Phase 7)"]),
        "spatial_max_auc"       : float(auc_results["Spatial Max (Phase 7)"]),
        "phase5_combined_auc"   : float(auc5_combined),
        "phase5_combined_cv"    : f"{cv5.mean():.4f}±{cv5.std():.4f}",
        "phase7_spatial_auc"    : float(auc7_combined),
        "phase7_spatial_cv"     : f"{cv7.mean():.4f}±{cv7.std():.4f}",
        "all_features_auc"      : float(aucAll_combined),
        "all_features_cv"       : f"{cvAll.mean():.4f}±{cvAll.std():.4f}",
    },
    "key_finding": (
        "Hallucinated images show LOWER spatial concentration (SCI) than clean images. "
        "Mode interpolation produces uniform global variance spread, not localized hot-spots. "
        "Geometric-only images show HIGH localized variance (SCI~0.58), forming a distinct "
        "third pattern. Spatial features provide complementary but not superior AUC vs Phase 5."
    )
}

with open("Phase7_Results/phase7_summary.json", "w") as f:
    json.dump(phase7_summary, f, indent=2)

print("Phase 7 summary saved.")
print(json.dumps(phase7_summary, indent=2))

Phase 7 summary saved.
{
  "spatial_concentration": {
    "hall_sci_mean": 0.38578280080301885,
    "clean_sci_mean": 0.43576390734450543,
    "geo_sci_mean": 0.4860496636454153,
    "cohens_d": -1.0135740515429692,
    "mannwhitney_p": 8.08420061542346e-147,
    "effect_r": 0.30809098217686937
  },
  "peak_variance_ratio": {
    "hall_pvr_mean": 3.332902170224472,
    "clean_pvr_mean": 4.165310753864771,
    "cohens_d": -0.7625047414656839,
    "mannwhitney_p": 2.0558595397666102e-148,
    "effect_r": 0.3097863401184635
  },
  "auc_comparison": {
    "global_variance_auc": 0.8765828824754834,
    "clip_score_auc": 0.8724684995949599,
    "sci_gini_auc": 0.8241836029804875,
    "peak_var_ratio_auc": 0.8259675143770244,
    "spatial_max_auc": 0.42114406738305077,
    "phase5_combined_auc": 0.8978637525470893,
    "phase5_combined_cv": "0.8962\u00b10.0478",
    "phase7_spatial_auc": 0.8994131527949933,
    "phase7_spatial_cv": "0.9000\u00b10.0796",
    "all_features_auc": 0.9258283259103

In [24]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from scipy.stats import gaussian_kde
from sklearn.metrics import roc_curve, roc_auc_score

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#0f0f0f')

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

ax1 = fig.add_subplot(gs[0, 0])   # Three-tier SCI bar chart
ax2 = fig.add_subplot(gs[0, 1])   # SCI vs Global Variance scatter
ax3 = fig.add_subplot(gs[0, 2])   # AUC comparison bar chart
ax4 = fig.add_subplot(gs[1, 0])   # CFG vs mean SCI line plot
ax5 = fig.add_subplot(gs[1, 1])   # ROC curves
ax6 = fig.add_subplot(gs[1, 2])   # Summary stats table

for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

# ── Panel 1: Three-tier SCI bar chart ────────────────────────────────────────
groups    = ['Hallucinated\n(both)', 'Semantic\nonly', 'Geometric\nonly', 'Clean\n(none)']
sci_means = [
    np.mean([m['sci_gini'] for m in smetrics if m['failure_type']=='both']),
    np.mean([m['sci_gini'] for m in smetrics if m['failure_type']=='semantic']),
    np.mean([m['sci_gini'] for m in smetrics if m['failure_type']=='geometric']),
    np.mean([m['sci_gini'] for m in smetrics if m['failure_type']=='none']),
]
sci_stds  = [
    np.std([m['sci_gini'] for m in smetrics if m['failure_type']=='both']),
    np.std([m['sci_gini'] for m in smetrics if m['failure_type']=='semantic']),
    np.std([m['sci_gini'] for m in smetrics if m['failure_type']=='geometric']),
    np.std([m['sci_gini'] for m in smetrics if m['failure_type']=='none']),
]
counts = [
    len([m for m in smetrics if m['failure_type']=='both']),
    len([m for m in smetrics if m['failure_type']=='semantic']),
    len([m for m in smetrics if m['failure_type']=='geometric']),
    len([m for m in smetrics if m['failure_type']=='none']),
]
colors_bar = ['#ff4444','#4488ff','#ff9900','#44cc88']

bars = ax1.bar(groups, sci_means, yerr=sci_stds, capsize=5,
               color=colors_bar, alpha=0.85,
               error_kw={'color':'white','elinewidth':1.5})
for bar, mean, count in zip(bars, sci_means, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, mean + 0.002,
             f'{mean:.3f}\n(n={count})',
             ha='center', va='bottom', color='white', fontsize=7.5)

ax1.set_ylabel('Mean SCI (Gini)', color='white', fontsize=10)
ax1.set_title('Three-Tier Spatial Pattern\nby Failure Type',
              color='white', fontsize=10, fontweight='bold')
ax1.set_ylim(0, 0.60)
ax1.tick_params(axis='x', labelsize=8)

# Annotation arrow: geometric > clean > semantic > hallucinated
ax1.annotate('Geometric-only has\nHIGHEST concentration\n(localized hot-spot)',
             xy=(2, sci_means[2]), xytext=(2.3, 0.54),
             arrowprops=dict(arrowstyle='->', color='#ff9900', lw=1.5),
             color='#ff9900', fontsize=7.5, ha='center')

# ── Panel 2: SCI vs Global Variance scatter (sampled) ────────────────────────
np.random.seed(42)
idx_sample = np.random.choice(len(smetrics), size=1500, replace=False)
sample     = [smetrics[i] for i in idx_sample]

type_colors = {'both':'#ff4444','semantic':'#4488ff',
               'geometric':'#ff9900','none':'#44cc88'}
type_alpha  = {'both':0.9,'semantic':0.3,'geometric':0.8,'none':0.2}
type_size   = {'both':20,'semantic':4,'geometric':15,'none':3}

for ftype, fc in type_colors.items():
    pts = [m for m in sample if m['failure_type']==ftype]
    if not pts: continue
    ax2.scatter([m['global_var'] for m in pts],
                [m['sci_gini']   for m in pts],
                c=fc, alpha=type_alpha[ftype],
                s=type_size[ftype], label=ftype, zorder=3)

ax2.set_xlabel('Global Variance', color='white', fontsize=10)
ax2.set_ylabel('SCI (Gini)', color='white', fontsize=10)
ax2.set_title('Global Variance vs Spatial Concentration\n(1500 sample)',
              color='white', fontsize=10, fontweight='bold')
ax2.legend(fontsize=7.5, facecolor='#2a2a2a', labelcolor='white',
           markerscale=2)

# Quadrant labels
ax2.axvline(0.003393, color='#888888', lw=0.8, ls='--', alpha=0.5)
ax2.axhline(0.436,    color='#888888', lw=0.8, ls='--', alpha=0.5)
ax2.text(0.016, 0.58, 'High var\nHigh SCI\n(geometric)', 
         color='#ff9900', fontsize=7, ha='center')
ax2.text(0.016, 0.22, 'High var\nLow SCI\n(hallucinated)',
         color='#ff4444', fontsize=7, ha='center')

# ── Panel 3: AUC comparison bar chart ────────────────────────────────────────
auc_names = ['Global\nVar', 'CLIP\nScore', 'SCI\nGini', 'PVR',
             'Ph5\nCombined', 'Ph7\nSpatial', 'All\nFeatures']
auc_vals  = [
    auc_results["Global Variance (Phase 5)"],
    auc_results["CLIP Score (Phase 5)"],
    auc_results["SCI / Gini (Phase 7)"],
    auc_results["Peak Variance Ratio (Phase 7)"],
    auc5_combined, auc7_combined, aucAll_combined
]
auc_colors = ['#ff4444','#4488ff','#ff9900','#aa44ff',
              '#ff6666','#ffaa44','#44ff88']

bars3 = ax3.bar(auc_names, auc_vals, color=auc_colors, alpha=0.85)
ax3.axhline(0.9, color='white', lw=0.8, ls=':', alpha=0.5,
            label='AUC=0.90 threshold')
ax3.axhline(0.8, color='#888888', lw=0.8, ls=':', alpha=0.4)
for bar, val in zip(bars3, auc_vals):
    ax3.text(bar.get_x() + bar.get_width()/2,
             val + 0.003, f'{val:.3f}',
             ha='center', va='bottom', color='white', fontsize=7.5)
ax3.set_ylim(0.35, 0.97)
ax3.set_ylabel('ROC-AUC', color='white', fontsize=10)
ax3.set_title('AUC by Signal & Combination\nPhase 5 vs Phase 7',
              color='white', fontsize=10, fontweight='bold')
ax3.legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white')
ax3.tick_params(axis='x', labelsize=8)

# ── Panel 4: CFG vs mean SCI per failure type ─────────────────────────────────
cfg_vals = sorted(set(m['cfg'] for m in smetrics))

for ftype, fc in type_colors.items():
    means = [np.mean([m['sci_gini'] for m in smetrics
                      if m['cfg']==c and m['failure_type']==ftype] or [np.nan])
             for c in cfg_vals]
    means_arr = np.array(means, dtype=float)
    valid = ~np.isnan(means_arr)
    if valid.sum() < 2: continue
    ax4.plot(np.array(cfg_vals)[valid], means_arr[valid],
             color=fc, lw=2, marker='o', markersize=5,
             label=ftype)

ax4.set_xlabel('CFG Scale', color='white', fontsize=10)
ax4.set_ylabel('Mean SCI (Gini)', color='white', fontsize=10)
ax4.set_title('SCI vs CFG Scale\nby Failure Type',
              color='white', fontsize=10, fontweight='bold')
ax4.legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white')

# ── Panel 5: ROC curves ───────────────────────────────────────────────────────
roc_configs = [
    ("Global Var",    X_gvar,   False, '#ff4444', '--', 2.0),
    ("CLIP Score",    X_clipsc, True,  '#4488ff', '--', 2.0),
    ("SCI (Ph7)",     X_sci,    True,  '#ff9900', '-',  2.0),
    ("All Combined",  XAll,     None,  '#44ff88', '-',  2.5),
]
for name, X, negate, color, ls, lw in roc_configs:
    if negate is None:
        proba = lrAll.predict_proba(X)[:,1]
        fpr, tpr, _ = roc_curve(y, proba)
        auc_v = roc_auc_score(y, proba)
    else:
        X_use = -X if negate else X
        fpr, tpr, _ = roc_curve(y, X_use)
        auc_v = roc_auc_score(y, X_use)
    ax5.plot(fpr, tpr, color=color, lw=lw, ls=ls,
             label=f"{name} (AUC={auc_v:.3f})")

ax5.plot([0,1],[0,1], color='#555555', lw=1, ls=':')
ax5.set_xlabel('False Positive Rate', color='white', fontsize=10)
ax5.set_ylabel('True Positive Rate', color='white', fontsize=10)
ax5.set_title('ROC Curves: Key Signals',
              color='white', fontsize=10, fontweight='bold')
ax5.legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white',
           loc='lower right')

# ── Panel 6: Summary stats table ─────────────────────────────────────────────
ax6.axis('off')
table_data = [
    ["Metric",                    "Value",       "Significance"],
    ["Hall. SCI mean",            "0.386",       "↓ vs clean (0.436)"],
    ["Geo-only SCI mean",         "0.486",       "↑ HIGHEST of all groups"],
    ["Cohen's d (SCI)",           "-1.014",      "Large effect"],
    ["Mann-Whitney p (SCI)",      "8.08e-147",   "***"],
    ["Cohen's d (PVR)",           "-0.763",      "Large effect"],
    ["AUC: Global Var alone",     "0.877",       "Phase 5 baseline"],
    ["AUC: SCI alone",            "0.824",       "Complementary signal"],
    ["AUC: All combined",         "0.926",       "+0.028 vs Phase 5"],
    ["5-fold CV (all features)",  "0.919±0.062", "Stable"],
]

col_colors = [['#2a3a4a']*3] + [['#1a1a1a']*3]*9
table = ax6.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='left', loc='center',
    cellColours=col_colors[1:],
    colColours=['#2a4a6a']*3
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.55)
for (row,col), cell in table.get_celld().items():
    cell.set_text_props(color='white')
    cell.set_edgecolor('#444444')

ax6.set_title('Phase 7 Key Statistics Summary',
              color='white', fontsize=10, fontweight='bold', pad=12)

fig.suptitle(
    'Phase 7 Complete Summary — Spatial Hallucination Analysis\n'
    'Mode interpolation produces UNIFORM variance spread; '
    'Geometric-only failures show LOCALIZED hot-spots',
    color='white', fontsize=13, fontweight='bold', y=0.98
)

plt.savefig("Phase7_Results/gallery/figure8_phase7_summary.png",
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.close()
print("Figure 8 saved → Phase7_Results/gallery/figure8_phase7_summary.png")

Figure 8 saved → Phase7_Results/gallery/figure8_phase7_summary.png
